# De un archivo a fragmentos con procedencia

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

| | |
|---|---|
| **Problema** | Un asistente que responde sobre los documentos de una organización no lee archivos: lee texto. Cada formato llega con su propio desorden, y un fragmento mal cortado separa una regla de lo que la hace entendible. |
| **Producto** | Una función que repara palabras cortadas, tres cargadores que producen la misma estructura (`Document`) y una decisión sobre el tamaño del fragmento. |
| **Recursos** | Sin modelos ni claves. La ejecución completa lleva unos segundos. |

Tintas Acme, una empresa ficticia del curso, quiere que su personal consulte las políticas internas. Las políticas están en un PDF, en un archivo Markdown y, en otros proyectos, en una tabla. Antes de buscar nada hay que dejar todo en la misma forma: texto, con su procedencia, en pedazos que un buscador pueda traer.

**¿Cómo convertimos un PDF, un Markdown y una tabla en la misma estructura, y cómo los partimos sin separar una regla de lo que la hace entendible?**

> **Cómo se trabaja.** En clase se presenta cada tema con el código ya escrito. Después lo explorás por tu cuenta: ejecutás las celdas, leés las notas **Para mirar** y cambiás valores para ver qué pasa. Al final de cada bloque hay un **microlaboratorio**: una tabla te dice qué líneas completar y con qué, y la celda siguiente comprueba si quedó bien.

## Los datos

| Campo | Dato |
|---|---|
| Fuentes | **Política interna** de Tintas Acme (PDF, 2 páginas) y **política de trabajo remoto** (Markdown): documentos ficticios creados para el curso. **Artículo de revista** del INTA (RIA, vol. 42, n.º 1, abril de 2016, pp. 19 a 22, PDF de 4 páginas): copia local de un texto real. **Fragmentos de cuentos** (CSV de la unidad 009, 137 filas). |
| Unidad de análisis | Un documento completo, que se convierte en fragmentos |
| Limitaciones | El PDF de Acme se generó por computadora y está escrito sin tildes; un PDF escaneado o con tablas se comporta distinto. El artículo del INTA es uno solo. |

In [1]:
from pathlib import Path

from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

import utils

utils.versiones(["langchain-core", "langchain-text-splitters", "pypdf"])

{'python': '3.12.13',
 'langchain-core': '1.7.0',
 'langchain-text-splitters': '1.1.3',
 'pypdf': '6.21.2'}

## 1. Un `Document`: texto más procedencia

LangChain representa cualquier fuente con un mismo objeto, `Document`, que tiene dos campos: `page_content` (el texto) y `metadata` (un diccionario con de dónde salió). Esa estructura es la que después recibe la base vectorial. Para un PDF usamos `pypdf`, que lee el texto de cada página, y armamos un `Document` por página. En `metadata` guardamos solo lo que vamos a usar para citar: el nombre del archivo y el número de página.

In [2]:
def cargar_pdf(ruta: Path, fuente: str) -> list[Document]:
    """Lee un PDF y devuelve un Document por pagina.

    Args:
        ruta: Ubicacion del archivo PDF.
        fuente: Nombre que se guarda como procedencia.

    Returns:
        Lista de Document con "fuente" y "pagina" (desde 1) como metadatos.
    """
    lector = PdfReader(str(ruta))
    paginas = []
    numero_de_pagina = 0
    for pagina in lector.pages:
        numero_de_pagina += 1
        paginas.append(
            Document(
                page_content=pagina.extract_text() or "",
                metadata={"fuente": fuente, "pagina": numero_de_pagina},
            )
        )
    return paginas


paginas_de_acme = cargar_pdf(
    utils.RUTA_DATOS / "politica_interna_tintas_acme.pdf",
    "politica_interna_tintas_acme.pdf",
)

print(f"Páginas cargadas: {len(paginas_de_acme)}")
primera = paginas_de_acme[0]
print("--- page_content (primeros 250 caracteres)")
print(primera.page_content[:250])
print("--- metadata")
print(primera.metadata)

Páginas cargadas: 2
--- page_content (primeros 250 caracteres)
POLITICA INTERNA DE RECURSOS HUMANOS, MATERIALES E INFORMACION
Tintas Acme S.A. — Version 1.0 — Junio 2026
1. SOLICITUD DE VACACIONES
Todo el personal de Tintas Acme tiene derecho a un minimo de quince dias habiles de
vacaciones anuales, de acuerdo a
--- metadata
{'fuente': 'politica_interna_tintas_acme.pdf', 'pagina': 1}


> **Para mirar.**
>
> - Hay un `Document` por página, no por sección: el título «2. SOLICITUD DE MATERIALES E INSUMOS» aparece en medio de la página 1.
> - Los renglones se cortan donde termina el ancho de la página, no donde termina una idea. Eso va a importar al buscar datos dentro del texto.
> - Las cadenas están escritas sin tildes porque el PDF de Acme así se generó.

## 2. Un PDF de revista

El PDF de Acme salió de una computadora y se lee limpio. Un artículo de revista está maquetado en columnas, con cabeceras, números de página y recuadros. Cargamos las cuatro páginas del INTA y miramos qué trae el texto extraído.

In [3]:
paginas_de_revista = cargar_pdf(
    utils.RUTA_DATOS / "ria_semillas_conservadas.pdf",
    "ria_semillas_conservadas.pdf",
)

print(f"Páginas: {len(paginas_de_revista)}")
print(paginas_de_revista[0].page_content[:700])

Páginas: 4
19
Abril 2016, Argentina
Semillas conservadas para un futuro sostenible
Semillas  conservadas para un 
futuro sostenible
Por Laura Pérez Casar
Legumbres
Destacadas por su gran aporte nutritivo, estos cultivos resultan esenciales 
para la seguridad alimentaria en un mundo cada vez más demandante de 
alimentos. El INTA preserva e investiga en el Banco de Germoplasma del NOA 
cerca de 700 entradas de porotos primitivos y silvestres. 
Mejorar la seguridad alimentaria y nu-
tricional de quienes habitan este mun-
do cada vez más poblado, es una prio-
ridad para los 42 países miembro de la 
Organización de las Naciones Unidas 
para la Alimentación y la Agricultura 
(FAO, por sus siglas en inglés). 


In [4]:
renglones_cortados = 0
renglones_totales = 0
for pagina in paginas_de_revista:
    for renglon in pagina.page_content.split("\n"):
        renglones_totales += 1
        if renglon.endswith("-"):
            renglones_cortados += 1

print("Renglones que terminan en guion:")
print(f"{renglones_cortados} de {renglones_totales}")

Renglones que terminan en guion:
84 de 338


> **Para mirar.**
>
> - 84 de 338 renglones del artículo terminan en guion: la palabra se corta en el borde de la columna.
> - La cabecera «ACTUALIDAD EN I+D RIA / Vol. 42 / N.º 1» y los números de página se mezclan con el texto, y el título se repite.

Un buscador por significado necesita palabras enteras: `nu-` y `tricional` son dos pedazos que no se parecen a «nutricional». Reparamos el caso más simple, la palabra cortada con guion al final del renglón. Es una **heurística**: también une mal las palabras compuestas que casualmente se cortan en el guion (`post-` y `guerra`), y la nota de límites debería decirlo.

In [5]:
texto_original = paginas_de_revista[0].page_content
posicion = texto_original.index("nu-")
print(repr(texto_original[posicion - 10:posicion + 30]))

'entaria y nu-\ntricional de quienes habit'


## 3. Un Markdown con títulos

Un Markdown trae su estructura escrita: los títulos (`#`, `##`) separan secciones. En lugar de tratar el archivo como una cadena, lo cortamos por títulos y guardamos el camino de títulos como metadato.

In [6]:
texto_remoto = (
    utils.RUTA_DATOS / "politica_trabajo_remoto_tintas_acme.md"
).read_text(encoding="utf-8")

cortador_de_titulos = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "titulo"), ("##", "seccion")]
)
secciones_remotas = cortador_de_titulos.split_text(texto_remoto)

filas_de_secciones = []
for seccion in secciones_remotas:
    seccion.metadata["fuente"] = "politica_trabajo_remoto_tintas_acme.md"
    filas_de_secciones.append({
        "sección": seccion.metadata.get("seccion", "(portada)"),
        "caracteres": len(seccion.page_content),
        "empieza con": seccion.page_content[:50].replace("\n", " "),
    })

utils.tabla(filas_de_secciones)

,sección,caracteres,empieza con
0,(portada),167,**Tintas Acme S.A.** **Versión 1.0 - Julio de 2026
1,1. Alcance,278,La modalidad de trabajo remoto está disponible par
2,2. Días permitidos y solicitud,373,Cada persona alcanzada puede realizar hasta dos dí
3,3. Horario y disponibilidad,208,"Durante la jornada remota, la persona debe permane"
4,4. Seguridad de la información,426,El trabajo remoto debe realizarse exclusivamente c
5,5. Excepciones,163,"Cualquier excepción a los días permitidos, los pue"


> **Para mirar.** Cada sección es un `Document` con su título en `metadata`. Una respuesta futura puede citar «sección 4. Seguridad de la información», algo que el PDF no permitía sin trabajo extra.

## 4. Una tabla

Una tabla ya viene cortada: cada fila es un fragmento. Lo que hay que decidir es qué columna es el texto y cuáles son metadatos. En los cuentos de la unidad 009, el texto es `texto` y el autor y el título permiten filtrar y citar.

In [7]:
cuentos = utils.cargar_cuentos()

documentos_de_cuentos = []
for posicion in range(len(cuentos)):
    fila = cuentos.iloc[posicion]
    documentos_de_cuentos.append(
        Document(
            page_content=fila["texto"],
            metadata={
                "id": fila["id"],
                "autor": fila["autor"],
                "titulo": fila["titulo"],
            },
        )
    )

print(f"Documentos: {len(documentos_de_cuentos)}")
print(documentos_de_cuentos[0].metadata)
print(documentos_de_cuentos[0].page_content[:150])

Documentos: 137
{'id': 'la-mosca-que-sonaba-que-era-un-aguila-01', 'autor': 'Augusto Monterroso', 'titulo': 'La mosca que soñaba que era un águila'}
Había una vez una Mosca que todas las noches soñaba que era un Águila y que se encontraba volando por los Alpes y por los Andes. En los primeros momen


> **Para mirar.** Los tres formatos terminaron en la misma estructura. Lo que cambia entre ellos es **cuánto trabajo costó** y **cuánta información de contexto quedó en `metadata`**: el PDF trae página, el Markdown trae sección, la tabla trae autor y título.

### Microlaboratorio 1 — Reparar las palabras cortadas

**Tarea.** Escribir una función que una las palabras que el PDF cortó con guion.  
**Producto.** `unir_palabras_cortadas`, comprobada con dos textos inventados y con el artículo real.  
**Antes de empezar.** Sección 2.

| Línea | Qué tiene que quedar | Con qué |
|---|---|---|
| (1) `texto_reparado` | el texto sin los cortes que traen un espacio antes del guion (`Germoplas -` y `ma`) | `replace` con `" -\n"` y la cadena vacía `""` |
| (2) `texto_reparado` | el texto sin los cortes comunes (`nu-` y `tricional`) | `replace` con `"-\n"` y la cadena vacía `""`, **después** de (1) |

**Comprobación.** `nu-\ntricional` tiene que dar `nutricional`, `Germoplas -\nma` tiene que dar `Germoplasma`, y el artículo ya no puede contener la secuencia guion más salto de renglón.

> **Ayuda 1.** En el PDF, un corte es el carácter `-` seguido de un salto de renglón, que en Python se escribe `\n`. Algunos tienen un espacio antes del guion.  
> **Ayuda 2.** `replace(buscado, reemplazo)` devuelve el texto con todas las apariciones cambiadas. Si quitás primero `"-\n"`, el espacio de `" -\n"` queda adentro de la palabra.

In [8]:
def unir_palabras_cortadas(texto: str) -> str:
    """Une las palabras que un PDF corta con guion al final de un renglon.

    Args:
        texto: Texto extraido de un PDF.

    Returns:
        El texto con las palabras cortadas unidas.
    """
    texto_reparado = texto  # COMPLETAR (1) y (2)
    return texto_reparado


utils.comprobar(
    unir_palabras_cortadas("nu-\ntricional") == "nutricional",
    "revisa (2): nu-\\ntricional tiene que quedar como nutricional",
)
utils.comprobar(
    unir_palabras_cortadas("Germoplas -\nma") == "Germoplasma",
    "revisa (1): Germoplas -\\nma tiene que quedar como Germoplasma",
)

reparadas = []
for pagina in paginas_de_revista:
    reparadas.append(unir_palabras_cortadas(pagina.page_content))
quedan = 0
for texto_de_pagina in reparadas:
    quedan += texto_de_pagina.count("-\n")
utils.comprobar(
    quedan == 0,
    "revisa (1) y (2): todavia quedan cortes con guion y salto de renglon",
)

Todavia no: revisa (2): nu-\ntricional tiene que quedar como nutricional
Todavia no: revisa (1): Germoplas -\nma tiene que quedar como Germoplasma
Todavia no: revisa (1) y (2): todavia quedan cortes con guion y salto de renglon


A partir de acá usamos `utils.unir_palabras_cortadas`, que hace lo mismo que la tuya, para que lo que sigue funcione aunque todavía no la hayas resuelto.

In [9]:
for pagina in paginas_de_revista:
    pagina.page_content = utils.unir_palabras_cortadas(pagina.page_content)

texto_reparado = paginas_de_revista[0].page_content
posicion = texto_reparado.index("nutricional")
print(repr(texto_reparado[posicion - 20:posicion + 40]))

'ridad alimentaria y nutricional de quienes habitan este mund'


> **Para mirar.**
>
> - «nutricional» quedó entera. Cuatro cortes traían un espacio antes del guion (`Germoplas -`): sin quitarlos primero, la palabra queda separada («Germoplas ma»).
> - Un corte cae al final de una página (`multiplica -`) y la palabra sigue en la siguiente; esta función no lo puede unir. Cada reparación es una heurística con límites, y conviene dejarlos escritos.

## 5. Cortar en fragmentos

Un modelo de embeddings y un modelo de lenguaje leen una cantidad limitada de texto, y un buscador funciona mejor con pedazos acotados. `RecursiveCharacterTextSplitter` corta un texto en fragmentos respetando, en este orden, saltos de párrafo, de renglón y espacios. Tiene dos parámetros que cambian el resultado:

| Parámetro | Qué controla | Qué cambia si lo modificás |
|---|---|---|
| `chunk_size` | largo máximo de cada fragmento, en caracteres | más grande: menos fragmentos y más contexto; más chico: cortes más finos pero reglas partidas |
| `chunk_overlap` | caracteres que se repiten entre un fragmento y el siguiente | más grande: menos riesgo de partir una idea, pero más texto repetido |

Probamos cuatro tamaños sobre el PDF de Acme. Para medir si el corte es bueno, miramos si **la regla de vacaciones queda entera en un mismo fragmento**: el plazo («treinta dias corridos»), el formulario (`RRHH-VAC-01`) y quién responde («cinco dias habiles de recibida»).

In [10]:
datos_de_la_regla = [
    "treinta dias corridos", "RRHH-VAC-01", "cinco dias habiles"
]


def contar_fragmentos_con_la_regla(fragmentos: list[Document]) -> int:
    """Cuenta los fragmentos que contienen los tres datos de la regla.

    Args:
        fragmentos: Fragmentos a revisar.

    Returns:
        Cantidad de fragmentos con los tres datos juntos.
    """
    cantidad = 0
    for fragmento in fragmentos:
        # Juntamos los renglones: el PDF corta donde termina el ancho.
        texto = " ".join(fragmento.page_content.split())
        completos = 0
        for dato in datos_de_la_regla:
            if dato in texto:
                completos += 1
        if completos == len(datos_de_la_regla):
            cantidad += 1
    return cantidad


filas_de_tamanos = []
for tamano in (100, 200, 500, 1000):
    solapamiento = tamano // 5
    divisor = RecursiveCharacterTextSplitter(
        chunk_size=tamano, chunk_overlap=solapamiento
    )
    fragmentos = divisor.split_documents(paginas_de_acme)
    largo_total = 0
    for fragmento in fragmentos:
        largo_total += len(fragmento.page_content)
    filas_de_tamanos.append({
        "chunk_size": tamano,
        "chunk_overlap": solapamiento,
        "fragmentos": len(fragmentos),
        "largo medio": round(largo_total / len(fragmentos)),
        "con la regla completa": contar_fragmentos_con_la_regla(fragmentos),
    })

utils.tabla(filas_de_tamanos)

,chunk_size,chunk_overlap,fragmentos,largo medio,con la regla completa
0,100,20,58,82,0
1,200,40,31,157,0
2,500,100,13,432,1
3,1000,200,6,908,1


> **Para mirar.**
>
> - Con 100 y 200 caracteres ningún fragmento contiene los tres datos juntos: la regla quedó partida (58 y 31 fragmentos).
> - Con 500 y 1000 hay un fragmento con la regla completa (13 y 6 fragmentos en total).
> - Más grande no es mejor sin medir: un fragmento de 1000 caracteres puede mezclar vacaciones con pedidos de materiales. Qué pasa al buscar con cada tamaño se mide en el cuaderno 04.

**Experimentar.** El solapamiento repite texto entre fragmentos vecinos. Antes de ejecutar, predecí: ¿qué pasa si pedimos más solapamiento que el tamaño del fragmento?

In [11]:
# error intencional: ValueError (el solapamiento no puede superar el tamano)
try:
    divisor_invalido = RecursiveCharacterTextSplitter(
        chunk_size=100, chunk_overlap=200
    )
except ValueError as error:
    print(f"ValueError: {error}")

ValueError: Got a larger chunk overlap (200) than chunk size (100), should be smaller.


La biblioteca rechaza la configuración: repetir más texto del que cabe en un fragmento no tiene sentido. Cuando falla así, avisa; lo que falla **sin avisar** es un tamaño razonable que parte una regla, como vimos en la tabla.

### Microlaboratorio 2 — ¿Qué tamaño conviene?

**Tarea.** Medir con otra regla del PDF qué tamaños la dejan entera, y decidir.  
**Producto.** `contar_fragmentos_con` y `mi_decision`.  
**Antes de empezar.** Sección 5.

La regla de insumos urgentes: la nota debe elevarse con **«cuarenta y ocho horas»** de anticipación y la gerencia responde en **«veinticuatro horas»**.

| Línea | Qué tiene que quedar | Con qué |
|---|---|---|
| (1) condición del `if` | verdadera si el dato aparece en el fragmento | `dato in texto`, donde `texto` ya tiene los renglones unidos |
| (2) `cantidad` | cuántos fragmentos tienen todos los datos | sumar `1` cuando `completos` es igual a `len(datos)` |
| Paso 3 | tu decisión en `mi_decision` | al menos 30 palabras que citen una cifra de la tabla |

> **Ayuda 1.** `contar_fragmentos_con_la_regla` de la sección 5 hace lo mismo con datos fijos.  
> **Ayuda 2.** Elegir el tamaño más grande no resuelve todo: el cuaderno 04 mide qué pasa en la recuperación.

In [12]:
def contar_fragmentos_con(fragmentos: list[Document], datos: list[str]) -> int:
    """Cuenta los fragmentos que contienen todos los datos pedidos.

    Args:
        fragmentos: Fragmentos a revisar.
        datos: Textos que tienen que aparecer juntos.

    Returns:
        Cantidad de fragmentos con todos los datos.
    """
    cantidad = 0
    for fragmento in fragmentos:
        texto = " ".join(fragmento.page_content.split())
        completos = 0
        for dato in datos:
            if False:  # COMPLETAR (1)
                completos += 1
        if completos == len(datos):
            cantidad += 0  # COMPLETAR (2)
    return cantidad


datos_de_insumos = ["cuarenta y ocho horas", "veinticuatro horas"]
cuentas = {}
for tamano in (100, 200, 500, 1000):
    divisor = RecursiveCharacterTextSplitter(
        chunk_size=tamano, chunk_overlap=tamano // 5
    )
    fragmentos = divisor.split_documents(paginas_de_acme)
    cuentas[tamano] = contar_fragmentos_con(fragmentos, datos_de_insumos)

utils.comprobar(
    cuentas[1000] >= 1 and cuentas[100] == 0,
    "revisa (1) y (2): con 1000 tiene que haber alguno y con 100, ninguno",
)
cuentas

mi_decision = ""
utils.comprobar(
    len(mi_decision.split()) >= 30,
    "escribi al menos 30 palabras en mi_decision",
)

Todavia no: revisa (1) y (2): con 1000 tiene que haber alguno y con 100, ninguno
Todavia no: escribi al menos 30 palabras en mi_decision


## Qué te llevás

- Cada fuente se convierte en `Document` con texto y procedencia. Lo que cambia entre PDF, Markdown y tabla es **cuánta estructura y cuánto desorden** traen; los metadatos son lo que después permite citar.
- Un PDF real necesita **reparaciones** antes de usarse (palabras cortadas, cabeceras, recuadros), y cada reparación es una heurística con límites.
- El tamaño del fragmento es una decisión con costo en los dos sentidos: **muy chico parte las reglas, muy grande mezcla temas**. No hay un valor correcto sin medir la recuperación.

**Sigue:** cómo se guardan los fragmentos para buscarlos por significado (cuaderno 02).